<b>Minimum Order: Discounts</b>

<img src="discount.jpg" width=55% align="left">

In [3]:
# data
MedThreshold = 1000
HighThreshold = 2500

Price = {(1,'Low') : 24, (1,'Medium') : 22, (1,'High') : 18,
         (2,'Low') : 20, (2,'Medium') : 18, (2,'High') : 16, 
         (3,'Low') : 22, (3,'Medium') : 19, (3,'High') : 16 }
Capacity = {1 : 5000, 2 : 3000, 3 : 4000}

Demand = 7500

Suppliers = Capacity.keys()
Tiers = {j for (i,j) in Price.keys()}

In [4]:
from docplex.mp.model import Model
mdl = Model()

In [5]:
# variables
amount = mdl.continuous_var_matrix(Suppliers, Tiers, lb=0, name='amount')
select = mdl.binary_var_matrix(Suppliers, Tiers, name='select')

In [6]:
# objective
mdl.minimize(mdl.sum(Price[i,j]*amount[i,j] for (i,j) in amount))

In [7]:
# constraints: min/max order to define the tiers
for i in Suppliers:
    mdl.add_constraint(amount[i,'Low'] <= (MedThreshold-1)*select[i,'Low'])
    mdl.add_constraint(amount[i,'Medium'] >= MedThreshold*select[i,'Medium'])
    mdl.add_constraint(amount[i,'Medium'] <= (HighThreshold-1)*select[i,'Medium'])
    mdl.add_constraint(amount[i,'High'] >= HighThreshold*select[i,'High'])
    mdl.add_constraint(amount[i,'High'] <= Capacity[i]*select[i,'High'])    

In [8]:
# select at most one tier per supplier
for i in Suppliers:
    mdl.add_constraint(mdl.sum(select[i,j] for j in Tiers) <= 1)

In [9]:
# constraint: meet demand
mdl.add_constraint(mdl.sum(amount[i,j] for (i,j) in amount) == Demand)

docplex.mp.LinearConstraint[](amount_1_Medium+amount_1_Low+amount_1_High+amount_2_Medium+amount_2_Low+amount_2_High+amount_3_Medium+amount_3_Low+amount_3_High,EQ,7500)

In [10]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.031,status='integer optimal solution')

In [11]:
mdl.print_solution()

objective: 124000.000
  amount_1_Low=500.000
  amount_2_High=3000.000
  amount_3_High=4000.000
  select_1_Low=1
  select_2_High=1
  select_3_High=1
